# Milestone 2: Data Cleaning and Preparation

In this notebook, I load the fictional retail data, inspect it, clean the data types, check missing values and duplicates, investigate unusual values, and create a few useful features.

**Important:** The retail dataset is synthetic. I will not remove returns or unusual sales automatically because they may be meaningful records.


## 1. Import libraries


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


## 2. Set the project folder and load the data

Update `PROJECT_DIR` if your project folder is saved somewhere else. 
This loads the sales-line file because it contains product quantities, prices, discounts and profit.


In [3]:
# Change this path if your project folder is in a different location.
PROJECT_DIR = Path.cwd().parent
DATA_FILE = PROJECT_DIR / 'data' / 'raw' / 'synthetic' / 'sales_lines_synthetic.csv'

sales_raw = pd.read_csv(DATA_FILE)

print('Rows and columns:', sales_raw.shape)
sales_raw.head()


Rows and columns: (175098, 17)


,transaction_id,line_number,transaction_date,store_id,customer_id,product_id,quantity,list_unit_price_kes,discount_pct,net_unit_price_kes,gross_sales_kes,net_sales_kes,unit_cost_kes,gross_profit_kes,promotion_id,transaction_status,is_synthetic
0,TX0000001,1,2025-01-10,ST005,C03186,SKU0165,1,"1,864.69",5,"1,771.46","1,864.69","1,771.46","1,403.23",368.23,NaN,SALE,True
1,TX0000001,2,2025-01-10,ST005,C03186,SKU0137,1,"5,030.84",10,"4,527.76","5,030.84","4,527.76","3,810.10",717.66,NaN,SALE,True
2,TX0000001,3,2025-01-10,ST005,C03186,SKU0127,1,"6,213.56",0,"6,213.56","6,213.56","6,213.56","3,345.95","2,867.61",NaN,SALE,True
3,TX0000001,4,2025-01-10,ST005,C03186,SKU0069,2,"6,242.04",5,"5,929.94","12,484.08","11,859.88","4,160.80","3,538.28",NaN,SALE,True
4,TX0000002,1,2025-12-21,ST004,C03906,SKU0214,1,"3,664.25",5,"3,481.04","3,664.25","3,481.04","1,806.68","1,674.36",NaN,SALE,True


## 3. Inspect the data before cleaning

These checks help me understand the columns and identify possible issues before changing anything.


In [4]:
def inspect_data(data):
    """
    Inspect a dataset's structure, summary statistics,
    missing values, and exact duplicate rows.
    """

    print("=" * 60, "1. FIRST FIVE ROWS", "=" * 60)
    display(data.head())

    print("\n" + "=" * 60, "2. DATASET INFORMATION", "=" * 60)
    data.info()

    print("\n" + "=" * 60, "3. DESCRIPTIVE STATISTICS", "=" * 60)
    display(data.describe(include="all").T)

    print("\n" + "=" * 60, "4. MISSING VALUES", "=" * 60)
    display(data.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

    print("\n" + "=" * 60, "5. EXACT DUPLICATE ROWS", "=" * 60, "=" * 60)
    print("Exact duplicate rows:", data.duplicated().sum())

In [5]:
inspect_data(sales_raw)

============================================================ 1. FIRST FIVE ROWS ============================================================


,transaction_id,line_number,transaction_date,store_id,customer_id,product_id,quantity,list_unit_price_kes,discount_pct,net_unit_price_kes,gross_sales_kes,net_sales_kes,unit_cost_kes,gross_profit_kes,promotion_id,transaction_status,is_synthetic
0,TX0000001,1,2025-01-10,ST005,C03186,SKU0165,1,"1,864.69",5,"1,771.46","1,864.69","1,771.46","1,403.23",368.23,NaN,SALE,True
1,TX0000001,2,2025-01-10,ST005,C03186,SKU0137,1,"5,030.84",10,"4,527.76","5,030.84","4,527.76","3,810.10",717.66,NaN,SALE,True
2,TX0000001,3,2025-01-10,ST005,C03186,SKU0127,1,"6,213.56",0,"6,213.56","6,213.56","6,213.56","3,345.95","2,867.61",NaN,SALE,True
3,TX0000001,4,2025-01-10,ST005,C03186,SKU0069,2,"6,242.04",5,"5,929.94","12,484.08","11,859.88","4,160.80","3,538.28",NaN,SALE,True
4,TX0000002,1,2025-12-21,ST004,C03906,SKU0214,1,"3,664.25",5,"3,481.04","3,664.25","3,481.04","1,806.68","1,674.36",NaN,SALE,True



============================================================ 2. DATASET INFORMATION ============================================================
<class 'pandas.DataFrame'>
RangeIndex: 175098 entries, 0 to 175097
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   transaction_id       175098 non-null  str    
 1   line_number          175098 non-null  int64  
 2   transaction_date     175098 non-null  str    
 3   store_id             175098 non-null  str    
 4   customer_id          175098 non-null  str    
 5   product_id           175098 non-null  str    
 6   quantity             175098 non-null  int64  
 7   list_unit_price_kes  175098 non-null  float64
 8   discount_pct         175098 non-null  int64  
 9   net_unit_price_kes   175098 non-null  float64
 10  gross_sales_kes      175098 non-null  float64
 11  net_sales_kes        175098 non-null  float64
 12  unit_cost_kes        175098 non-nul

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,175098,80000,TX0000030,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line_number,"175,098.00",NaN,NaN,NaN,1.90,1.03,1.00,1.00,2.00,2.00,5.00
transaction_date,175098,730,2025-08-27,310,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_id,175098,8,ST002,22376,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,175098,4000,C02444,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,175098,250,SKU0184,779,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,"175,098.00",NaN,NaN,NaN,1.35,0.85,-4.00,1.00,1.00,2.00,4.00
list_unit_price_kes,"175,098.00",NaN,NaN,NaN,"3,369.29","1,727.74",190.95,"1,929.94","3,345.54","4,776.53","6,431.72"
discount_pct,"175,098.00",NaN,NaN,NaN,3.01,4.01,0.00,0.00,0.00,5.00,25.00
net_unit_price_kes,"175,098.00",NaN,NaN,NaN,"3,268.25","1,683.49",171.85,"1,868.55","3,252.29","4,619.46","6,431.72"



============================================================ 4. MISSING VALUES ============================================================


,missing_values
promotion_id,175062
transaction_id,0
line_number,0
store_id,0
transaction_date,0
product_id,0
quantity,0
list_unit_price_kes,0
customer_id,0
discount_pct,0



============================================================ 5. EXACT DUPLICATE ROWS ============================================================ ============================================================
Exact duplicate rows: 0


## 4. Create a reusable cleaning function

The function below works on a copy, converts important columns to the correct types, removes only exact duplicate rows, standardises text, and creates a report of missing values and unusual values. It does **not** automatically remove outliers or negative quantities.


In [ ]:
# Create a copy of the raw data to work with
df = sales_raw.copy()

#### Step 1: Standardise column names and text fields

In [14]:
def standardise_text(df):
    # Standardise column names
    df.columns = (df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True))

    # Standardise text columns
    text_columns = df.select_dtypes(include=['object', 'string']).columns
    for col in text_columns:
        df[col] = df[col].astype(str).str.strip()

    return df


#### Step 2: Convert dates and numeric fields

In [10]:
def convert_data_types(df):

# convert transaction date
    if 'transaction_date' in df.columns:
        df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

# Convert numeric columns
    numeric_columns = [
            "line_number",
            "quantity",
            "list_unit_price_kes",
            "discount_pct",
            "net_unit_price_kes",
            "gross_sales_kes",
            "net_sales_kes",
            "unit_cost_kes",
            "gross_profit_kes",
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df   

#### Step 3: Remove only exact duplicate rows

In [11]:
def duplicates(df):

    duplicate_count = int(df.duplicated().sum())

    df = df.drop_duplicates()

    return df, duplicate_count


#### Step 4: Standardise identifiers and categories

In [12]:
def standardise_identifiers(df):
    # Standardize identifiers
    identifier_columns = [
        "transaction_id",
        "store_id",
        "customer_id",
        "product_id",
        "promotion_id",
    ]

    for col in identifier_columns:
        if col in df.columns:
            df[col] = df[col].replace("", pd.NA)

    return df

#### Step 5: Report unusual values

In [13]:
def unusual_values(df):
    issues = {}

    if "quantity" in df.columns:
        issues["negative_quantity_rows"] = int((df["quantity"] < 0).sum())

    if "discount_pct" in df.columns:
        issues["discount_below_0_or_above_100"] = int(((df["discount_pct"] < 0) | (df["discount_pct"] > 100)).sum())

    for column in [
        "list_unit_price_kes",
        "net_unit_price_kes",
        "unit_cost_kes",
    ]:
        if column in df.columns:
            issues[f"{column}_negative_rows"] = int((df[column] < 0).sum())

    if "transaction_status" in df.columns:
        valid_statuses = ["SALE", "RETURN", "VOID"]

        issues["unknown_status_rows"] = int((df["transaction_status"].notna()
                & ~df["transaction_status"].isin(valid_statuses)).sum())
        
    return issues

#### Step 6: Combine the five functions

In [27]:
def clean_sales_data(df):
    """Run the complete sales-data cleaning workflow."""

    before_count = len(df)

    # 1. Standardise column names and text
    df = standardise_text(df)

    # 2. Convert dates and numeric fields
    df = convert_data_types(df)

    # 3. Remove only exact duplicates
    df, duplicate_count = duplicates(df)

    # 4. Standardise identifiers
    df = standardise_identifiers(df)

    # 5. Report unusual values
    issues = unusual_values(df)

    # Create cleaning report
    report = {
        "rows_before": int(before_count),
        "rows_after": int(len(df)),
        "exact_duplicate_rows_removed": duplicate_count,
        "missing_values_by_column": df.isna().sum().to_dict(),
        "unusual_values_to_review": issues,
    }

    return df, report

#### Step 7: Run and inspect the results

In [28]:
sales_clean, cleaning_report = clean_sales_data(df)

print("Cleaning report:")
print(cleaning_report)

display(sales_clean.head())
display(sales_clean.isna().sum().to_frame("missing_values"))

Cleaning report:
{'rows_before': 175098, 'rows_after': 175098, 'exact_duplicate_rows_removed': 0, 'missing_values_by_column': {'transaction_id': 0, 'line_number': 0, 'transaction_date': 0, 'store_id': 0, 'customer_id': 0, 'product_id': 0, 'quantity': 0, 'list_unit_price_kes': 0, 'discount_pct': 0, 'net_unit_price_kes': 0, 'gross_sales_kes': 0, 'net_sales_kes': 0, 'unit_cost_kes': 0, 'gross_profit_kes': 0, 'promotion_id': 175062, 'transaction_status': 0, 'is_synthetic': 0}, 'unusual_values_to_review': {'negative_quantity_rows': 4430, 'discount_below_0_or_above_100': 0, 'list_unit_price_kes_negative_rows': 0, 'net_unit_price_kes_negative_rows': 0, 'unit_cost_kes_negative_rows': 0, 'unknown_status_rows': 0}}


,transaction_id,line_number,transaction_date,store_id,customer_id,product_id,quantity,list_unit_price_kes,discount_pct,net_unit_price_kes,gross_sales_kes,net_sales_kes,unit_cost_kes,gross_profit_kes,promotion_id,transaction_status,is_synthetic
0,TX0000001,1,2025-01-10,ST005,C03186,SKU0165,1,"1,864.69",5,"1,771.46","1,864.69","1,771.46","1,403.23",368.23,NaN,SALE,True
1,TX0000001,2,2025-01-10,ST005,C03186,SKU0137,1,"5,030.84",10,"4,527.76","5,030.84","4,527.76","3,810.10",717.66,NaN,SALE,True
2,TX0000001,3,2025-01-10,ST005,C03186,SKU0127,1,"6,213.56",0,"6,213.56","6,213.56","6,213.56","3,345.95","2,867.61",NaN,SALE,True
3,TX0000001,4,2025-01-10,ST005,C03186,SKU0069,2,"6,242.04",5,"5,929.94","12,484.08","11,859.88","4,160.80","3,538.28",NaN,SALE,True
4,TX0000002,1,2025-12-21,ST004,C03906,SKU0214,1,"3,664.25",5,"3,481.04","3,664.25","3,481.04","1,806.68","1,674.36",NaN,SALE,True


,missing_values
transaction_id,0
line_number,0
transaction_date,0
store_id,0
customer_id,0
product_id,0
quantity,0
list_unit_price_kes,0
discount_pct,0
net_unit_price_kes,0


## 5. Review missing values and duplicates

I will review each column before deciding whether to fill, keep or remove missing values.


In [21]:
missing_report = sales_clean.isna().sum().sort_values(ascending=False).to_frame('missing_count')
missing_report['missing_percent'] = (missing_report['missing_count'] / len(sales_clean) * 100).round(2)
display(missing_report)

print('Remaining exact duplicate rows:', sales_clean.duplicated().sum())
print('Duplicate transaction-line keys:', sales_clean.duplicated(['transaction_id', 'line_number']).sum())


,missing_count,missing_percent
promotion_id,175062,99.98
transaction_id,0,0.00
line_number,0,0.00
store_id,0,0.00
transaction_date,0,0.00
product_id,0,0.00
quantity,0,0.00
list_unit_price_kes,0,0.00
customer_id,0,0.00
discount_pct,0,0.00


Remaining exact duplicate rows: 0
Duplicate transaction-line keys: 0


## 6. Check unusual values and possible outliers

I use the IQR method to flag unusually high or low values. A flagged value is only a value to investigate; it is not automatically wrong. Negative quantities are checked separately because they can represent returns.


In [22]:
def flag_iqr_outliers(df, column):
    #Return rows outside 1.5 x IQR fences for one numeric column.
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = df[(df[column] < lower) | (df[column] > upper)].copy()
    return flagged, {'column': column, 'q1': q1, 'q3': q3, 'iqr': iqr,
                     'lower_fence': lower, 'upper_fence': upper,
                     'flagged_rows': len(flagged)}

price_outliers, price_outlier_summary = flag_iqr_outliers(sales_clean, 'net_sales_kes')
print(price_outlier_summary)
display(price_outliers.head(10))

# Check the types of transactions separately.
display(sales_clean['transaction_status'].value_counts(dropna=False).to_frame('row_count'))
print('Negative quantity rows (possible returns):', (sales_clean['quantity'] < 0).sum())


{'column': 'net_sales_kes', 'q1': np.float64(2065.21), 'q3': np.float64(5637.99), 'iqr': np.float64(3572.7799999999997), 'lower_fence': np.float64(-3293.96), 'upper_fence': np.float64(10997.16), 'flagged_rows': 14212}


,transaction_id,line_number,transaction_date,store_id,customer_id,product_id,quantity,list_unit_price_kes,discount_pct,net_unit_price_kes,gross_sales_kes,net_sales_kes,unit_cost_kes,gross_profit_kes,promotion_id,transaction_status,is_synthetic
3,TX0000001,4,2025-01-10,ST005,C03186,SKU0069,2,"6,242.04",5,"5,929.94","12,484.08","11,859.88","4,160.80","3,538.28",NaN,SALE,True
21,TX0000010,3,2025-01-07,ST001,C01193,SKU0206,3,"5,342.39",0,"5,342.39","16,027.17","16,027.17","2,687.05","7,966.02",NaN,SALE,True
36,TX0000017,2,2025-05-10,ST005,C02260,SKU0074,3,"4,929.60",0,"4,929.60","14,788.80","14,788.80","2,894.63","6,104.91",NaN,SALE,True
40,TX0000020,2,2025-09-30,ST004,C02989,SKU0127,2,"6,213.56",5,"5,902.88","12,427.12","11,805.76","3,345.95","5,113.86",NaN,SALE,True
46,TX0000024,2,2024-12-31,ST003,C01736,SKU0038,3,"5,720.38",0,"5,720.38","17,161.14","17,161.14","2,829.03","8,674.05",NaN,SALE,True
60,TX0000030,2,2026-07-20,ST003,C03612,SKU0152,2,"5,546.40",0,"5,546.40","11,092.80","11,092.80","3,850.08","3,392.64",NaN,SALE,True
62,TX0000030,4,2026-07-20,ST003,C03612,SKU0008,2,"6,183.63",5,"5,874.45","12,367.26","11,748.90","3,066.28","5,616.34",NaN,SALE,True
77,TX0000038,1,2025-07-17,ST007,C03905,SKU0164,4,"3,732.99",5,"3,546.34","14,931.96","14,185.36","2,272.98","5,093.44",NaN,SALE,True
78,TX0000039,1,2024-11-10,ST003,C03186,SKU0069,3,"6,242.04",0,"6,242.04","18,726.12","18,726.12","4,160.80","6,243.72",NaN,SALE,True
86,TX0000041,2,2025-05-02,ST006,C00261,SKU0222,3,"4,161.47",0,"4,161.47","12,484.41","12,484.41","2,799.37","4,086.30",NaN,SALE,True


,row_count
transaction_status,
SALE,168859
RETURN,4430
VOID,1809


Negative quantity rows (possible returns): 4430


## 7. Create simple features

These columns make later analysis easier. The date features help compare sales by month and weekday. The discount group is a simple label for describing discounts.


In [23]:
sales_clean['year'] = sales_clean['transaction_date'].dt.year
sales_clean['month'] = sales_clean['transaction_date'].dt.month
sales_clean['year_month'] = sales_clean['transaction_date'].dt.to_period('M').astype('string')
sales_clean['day_name'] = sales_clean['transaction_date'].dt.day_name()

sales_clean['discount_group'] = pd.cut(
    sales_clean['discount_pct'],
    bins=[-0.01, 0, 10, 20, 100],
    labels=['No discount', 'Low (above 0 to 10%)', 'Medium (above 10 to 20%)', 'High (above 20%)']
)

display(sales_clean[['transaction_date', 'year_month', 'day_name', 'discount_pct', 'discount_group']].head())


,transaction_date,year_month,day_name,discount_pct,discount_group
0,2025-01-10,2025-01,Friday,5,Low (above 0 to 10%)
1,2025-01-10,2025-01,Friday,10,Low (above 0 to 10%)
2,2025-01-10,2025-01,Friday,0,No discount
3,2025-01-10,2025-01,Friday,5,Low (above 0 to 10%)
4,2025-12-21,2025-12,Sunday,5,Low (above 0 to 10%)


## 8. Final checks and save the cleaned data

Before saving, I check the row count, date range, duplicate line keys and basic value rules. I keep the raw file unchanged and save the cleaned version separately.


In [26]:
print('Cleaned shape:', sales_clean.shape)
print('Date range:', sales_clean['transaction_date'].min(), 'to', sales_clean['transaction_date'].max())
print('Duplicate transaction-line keys:', sales_clean.duplicated(['transaction_id', 'line_number']).sum())
print('Invalid discounts:', ((sales_clean['discount_pct'] < 0) | (sales_clean['discount_pct'] > 100)).sum())
print('Missing transaction IDs:', sales_clean['transaction_id'].isna().sum())

OUTPUT_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'sales_lines_clean.csv'
sales_clean.to_csv(OUTPUT_FILE, index=False)
print('Saved cleaned data to:', OUTPUT_FILE)


Cleaned shape: (175098, 22)
Date range: 2024-10-01 00:00:00 to 2026-09-30 00:00:00
Duplicate transaction-line keys: 0
Invalid discounts: 0
Missing transaction IDs: 0
Saved cleaned data to: c:\Users\jndalamia\Documents\DataScience\customer-sales-intelligence-kenya\data\processed\sales_lines_clean.csv


In [25]:
print('Cleaned shape:', sales_raw.shape)
print('Date range:', sales_raw['transaction_date'].min(), 'to', sales_raw['transaction_date'].max())
print('Duplicate transaction-line keys:', sales_raw.duplicated(['transaction_id', 'line_number']).sum())
print('Invalid discounts:', ((sales_raw['discount_pct'] < 0) | (sales_raw['discount_pct'] > 100)).sum())
print('Missing transaction IDs:', sales_raw['transaction_id'].isna().sum())

Cleaned shape: (175098, 17)
Date range: 2024-10-01 to 2026-09-30
Duplicate transaction-line keys: 0
Invalid discounts: 0
Missing transaction IDs: 0
